# Simple RAG — RAG-EDU-1
Educational document-based question-answering system with answer generation.

Ushbu notebook TS (texnik topshiriq) asosida quyidagi 2 ta bosqichni amalga oshiradi:
1. **Hujjatlarni indekslash** (bir marta bajariladi)
2. **Savolga javob berish** (har bir so'rovda bajariladi)

Namunaviy hujjat sifatida `PF-140` (Prezident Farmoni, sudlarda sun'iy intellekt) fayli ishlatiladi.

**Muhim:** agar hujjatlarni (documents papkasi tarkibini) o'zgartirsangiz yoki quyidagi parametrlarni (chunk_size, overlap) o'zgartirsangiz, **1-bosqichdan (bazani tozalash) boshlab, barcha kataklarni qayta, ketma-ket ishga tushiring**. Aks holda baza eski ma'lumot bilan aralashib qoladi.

## Bosqich 0 — kutubxonalarni o'rnatish va API kalitni sozlash

Bu bo'limda, biz AI tizimimizni ishlashi uchun zarur bo'lgan dasturiy ta'minotlarni o'rnatamiz va unga maxsus 'kalit' (API key) beramiz. Bu kalit tizimimizga OpenAI kabi kuchli sun'iy intellekt xizmatlaridan foydalanishga ruxsat beradi. Xavfsizlik uchun bu kalitni maxfiy saqlash muhimdir.

Bu buyruq AI tizimimizga kerak bo'ladigan asosiy dasturlarni (kutubxonalarni) o'rnatadi. Tasavvur qiling, bu sizning ish stolingizga yangi yuridik dasturlarni o'rnatishga o'xshaydi. 'openai' kutubxonasi AI bilan muloqot qilishga, 'chromadb' esa hujjatlarimizni samarali saqlash va qidirishga yordam beradi.

In [9]:
!pip install -q openai chromadb

In [10]:
import os
from getpass import getpass

# OpenAI API kalitingizni kiriting (ekranda ko'rinmaydi)
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

from openai import OpenAI
client = OpenAI()

EMBEDDING_MODEL = "text-embedding-3-small"
GENERATION_MODEL = "gpt-4o-mini"
CHUNK_SIZE = 250
CHUNK_OVERLAP = 100
TOP_K = 8

def embed_texts(texts):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


OpenAI API key: ··········


Bu qismda, biz tizimimizni sun'iy intellekt xizmatlariga ulaymiz. Siz kiritgan 'OpenAI API key' — bu AI tizimlariga kirish uchun shaxsiy ruxsatnomangiz. U xavfsizlik nuqtai nazaridan yashiriladi va faqat sizning loyihangizga AI resurslaridan foydalanishga imkon beradi. Shuningdek, bu yerda hujjatlarni AI 'tushunishi' uchun ularni raqamli kodlarga aylantiruvchi maxsus vositalar sozlanadi.

## Bosqich 1 — hujjatlarni yuklash



Bu bosqichda biz AI tizimimizga tahlil qilishi kerak bo'lgan yuridik hujjatlarni taqdim etamiz. Siz o'zingizning qonunlar, farmonlar yoki sud qarorlaringizni 'documents' nomli papkaga joylashtirishingiz mumkin. Tizim bu fayllarni o'qib, ularni keyingi ishlov berishga tayyorlaydi.

Ushbu kod, 'documents' papkasidagi barcha matn (.txt yoki .md) fayllarni avtomatik tarzda topadi va ularni o'qiydi. Bu xuddi yuridik assistentingizning barcha jildlarni ochib, ichidagi hujjatlarni varaqlab chiqishiga o'xshaydi. Natijada, tizim qancha hujjat topganini va ularning hajmini sizga xabar beradi.

In [11]:
import glob

DOCS_DIR = "documents"  # shu papkaga o'z fayllaringizni joylang
os.makedirs(DOCS_DIR, exist_ok=True)

documents = {}
for path in glob.glob(f"{DOCS_DIR}/*.txt") + glob.glob(f"{DOCS_DIR}/*.md"):
    with open(path, "r", encoding="utf-8") as f:
        documents[os.path.basename(path)] = f.read()

print(f"Yuklangan hujjatlar soni: {len(documents)}")
for name, text in documents.items():
    print(f" - {name}: {len(text)} belgi")


Yuklangan hujjatlar soni: 1
 - PF-140.txt: 43596 belgi


## Bosqich 2 — vector database'ni tozalash va tayyorlash
Bu katakni har safar hujjatlar yoki parametrlar o'zgarganda ishga tushiring — eski ma'lumot bilan aralashib ketmasligi uchun.

Bu bosqich AI tizimimizning ma'lumotlar bazasini yangilash va tozalash uchun juda muhimdir. Agar siz hujjatlarni o'zgartirgan bo'lsangiz yoki sozlamalarni yangilagan bo'lsangiz, eski ma'lumotlar yangilariga aralashib ketmasligi uchun bu bosqichni qayta ishga tushirish kerak.


Bu yerda biz 'vector database' deb ataladigan maxsus ma'lumotlar bazasini sozlaymiz. U xuddi yuridik hujjatlaringiz uchun juda tezkor va aqlli indeks yaratuvchi raqamli kartotekaga o'xshaydi. Avvalo, biz eski indeksni o'chirib tashlaymiz (agar mavjud bo'lsa), keyin esa yangi, toza indeksni yaratamiz, bu tizimning samarali ishlashini ta'minlaydi.

In [12]:
import chromadb

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("rag_edu_1")
except Exception:
    pass  # baza hali mavjud bo'lmasa, xato e'tiborga olinmaydi

collection = chroma_client.get_or_create_collection(name="rag_edu_1")
print("Baza tozalandi va tayyor.")


Baza tozalandi va tayyor.


## Bosqich 3 — hujjatlarni fragmentlarga (chunk) bo'lish

Uzun yuridik hujjatlar ba'zan bitta savolga javob berish uchun juda katta bo'lishi mumkin. Bu bosqichda biz har bir hujjatni kichikroq, boshqariladigan 'fragmentlarga' (chunklarga) bo'lib chiqamiz. Bu jarayon AIga ma'lumotlarni tezroq topish va savolingizga tegishli qismlarga e'tibor qaratishga yordam beradi.

Bu kod funksiyasi uzun yuridik matnlarni qismlarga ajratishni amalga oshiradi. Har bir katta hujjat belgilangan o'lchamdagi kichik bo'laklarga bo'linadi, bu AIga aniq ma'lumotlarni topishni osonlashtiradi. Har bir bo'lak qaysi hujjatdan olinganligi va uning boshlang'ich tartib raqami bilan birga saqlanadi.

In [13]:
def split_into_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

all_chunks = []       # matn fragmentlari
all_metadatas = []    # har bir fragment qaysi fayldan va qaysi tartibda olinganini saqlaydi
all_ids = []

for doc_name, text in documents.items():
    chunks = split_into_chunks(text)
    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadatas.append({"source": doc_name, "chunk_index": i})
        all_ids.append(f"{doc_name}_{i}")

print(f"Jami fragmentlar soni: {len(all_chunks)}")


Jami fragmentlar soni: 291


## Bosqich 4 — vektorlashtirish va vector database'ga saqlash

Hujjat fragmentlari tayyor bo'lgandan so'ng, ularni AI 'tushunadigan' tilga aylantirish kerak. Bu bosqichda har bir fragmentni raqamli ko'rinishga (vektorlarga) o'tkazamiz va ularni maxsus ma'lumotlar bazamizga (vector database) joylashtiramiz. Bu jarayon AIga savolingizga eng mos keladigan ma'lumotlarni tezda topishga imkon beradi.

Bu kod har bir hujjat fragmentini 'vektor' deb ataladigan noyob raqamli 'barmoq iziga' aylantiradi. Keyin bu 'barmoq izlari' bizning raqamli kartotekamizga (ChromaDB) saqlanadi. Natijada, tizim savolingizni ham xuddi shunday 'barmoq iziga' aylantirib, minglab fragmentlar ichidan eng tez va aniq javobni topa oladi.

In [14]:
BATCH = 100
for i in range(0, len(all_chunks), BATCH):
    batch_chunks = all_chunks[i:i+BATCH]
    batch_ids = all_ids[i:i+BATCH]
    batch_meta = all_metadatas[i:i+BATCH]
    embeddings = embed_texts(batch_chunks)
    collection.add(
        ids=batch_ids,
        embeddings=embeddings,
        documents=batch_chunks,
        metadatas=batch_meta,
    )

print("Indekslash yakunlandi. Bazadagi fragmentlar soni:", collection.count())


Indekslash yakunlandi. Bazadagi fragmentlar soni: 291


## Bosqich 5 — so'rovni qayta ishlash (retrieval + generation)

Endi tizimimiz savollarga javob berishga tayyor. Bu bosqichda sizning savolingizni tahlil qilib, unga tegishli ma'lumotlarni topamiz (retrieval) va topilgan ma'lumotlar asosida tushunarli, aniq javobni shakllantiramiz (generation). Bu xuddi tajribali yuristning qonunlarni ko'rib chiqib, sizning holatingizga mos javobni topishiga o'xshaydi.

Ushbu funksiya sizning yuridik savolingizga javob berish uchun butun jarayonni boshqaradi. U savolingizni AI tushunadigan shaklga o'tkazadi, vektor bazadan eng tegishli hujjat qismlarini izlab topadi va ulardan foydalanib javob yaratadi. Eng muhimi, u faqat taqdim etilgan hujjatlarga asoslangan holda javob berishga o'rgatilgan, bu esa 'o'ylab topilgan' ma'lumotlarning oldini oladi. Natijada, sizga javob va uning qaysi manbalardan olinganligi ko'rsatiladi.

In [16]:
def answer_question(question, top_k=TOP_K, verbose=True):
    # 1) savolni vektorlashtirish
    q_embedding = embed_texts([question])[0]

    # 2) eng yaqin fragmentlarni topish
    results = collection.query(query_embeddings=[q_embedding], n_results=top_k)
    docs = results["documents"][0]
    metas = results["metadatas"][0]

    # 3) fragmentlarni hujjat ichidagi asl tartibga solish (kontekst uzuq-yuluq bo'lib qolmasligi uchun)
    combined = sorted(zip(docs, metas), key=lambda x: (x[1]["source"], x[1]["chunk_index"]))
    fragments = [c[0] for c in combined]
    sources = [c[1]["source"] for c in combined]

    if verbose:
        print("Topilgan manbalar:", set(sources))

    # 4) prompt yig'ish
    context = "\n\n---\n\n".join(fragments)
    system_prompt = (
        """Siz faqat quyida berilgan kontekst asosida javob beruvchi yordamchisiz. "
        "Kontekst bir necha alohida qonun bandlaridan iborat bo'lishi mumkin — ularni diqqat bilan o'qing, "
        "chunki javob bir necha qo'shni band yoki jumlalarni birlashtirib o'qiganda ma'lum bo'lishi mumkin. "
        "So'zlar sinonim yoki boshqacha shaklda kelishi mumkin — mazmuniga qarab javob bering. "
        "Faqat kontekstda haqiqatan ham hech qanday tegishli ma'lumot bo'lmagandagina "
        "'Berilgan hujjatlarda bu haqda ma'lumot topilmadi' deb javob bering."""
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Kontekst: {context}\nSavol: {question}"}
    ]
    response = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=messages
    )
    answer = response.choices[0].message.content
    return answer, list(set(sources)) # Return answer and unique sources

## Bosqich 6 — qabul qilish testlari (TS Section 6)
TS'ning 6-bo'limiga muvofiq quyidagi tekshiruvlarni bajaring:
1. Ma'lum javobli savol — tegishli fragment topiladimi?
2. Javob hujjat mazmuniga mos keladimi (o'ylab topilmaganmi)?
3. Hujjatda yo'q savolga — tizim "ma'lumot yo'q" deb javob beradimi?
4. Javobda manba ko'rsatilganmi?
5. Bir xil savolni qayta yuborganda natija barqarormi?

Har qanday yuridik tizim kabi, AI yordamchisi ham to'g'ri ishlashini tekshirish uchun sinovdan o'tkazilishi kerak. Bu bo'limda biz tizimga turli xil savollar beramiz, jumladan, aniq javobi bor savollar va hujjatlarda yo'q savollar. Maqsad – tizimning javoblari qanchalik aniq va ishonchli ekanligini, shuningdek, u ma'lumot bo'lmaganda 'bilmayman' deyishni bilishini tekshirish.

Bu kod qismi avvaldan tayyorlangan test savollar ro'yxatini oladi va har bir savolga AI tizimimizdan javob so'raydi. U har bir savol uchun javobni va javob olingan manbalarni chop etadi. Bu xuddi yuridik ekspertning hujjatlar bo'yicha bir qancha savollarga javob berishini kuzatishga o'xshaydi, bu esa tizimning ish faoliyatini baholashga yordam beradi.

In [17]:
test_questions = [
    "Iqtisodiy sudlar tuzilmasi qanday optimallashtiriladi?",
    "Toshkent shahrida qachon 'Raqamli sud' zallari tashkil etiladi?",
    "Sud majlisi bayonnomalari qanday tayyorlanadi?",
    "Marsda sud tizimi qanday ishlaydi?",  # hujjatda yo'q — "topilmadi" bo'lishi kerak
]

for q in test_questions:
    a, s = answer_question(q, verbose=False)
    print(f"SAVOL: {q}\nJAVOB: {a}\nMANBA: {s}\n{'-'*60}")


SAVOL: Iqtisodiy sudlar tuzilmasi qanday optimallashtiriladi?
JAVOB: Iqtisodiy sudlar tuzilmasi tumanlararo, tuman va shahar iqtisodiy sudlarini tugatish orqali maqbullashtiriladi. Bu chora-tadbir Oliy sud, Sudyalar oliy kengashi va Savdo-sanoat palatasi tomonidan amalga oshirilishi rejalashtirilmoqda.
MANBA: ['PF-140.txt']
------------------------------------------------------------
SAVOL: Toshkent shahrida qachon 'Raqamli sud' zallari tashkil etiladi?
JAVOB: Toshkent shahrida "Raqamli sud" zallari 2025-yil yakuniga qadar tajriba tariqasida tashkil etiladi.
MANBA: ['PF-140.txt']
------------------------------------------------------------
SAVOL: Sud majlisi bayonnomalari qanday tayyorlanadi?
JAVOB: Sud majlisi bayonnomalari real vaqt rejimida matn shaklida tayyorlanadi.
MANBA: ['PF-140.txt']
------------------------------------------------------------
SAVOL: Marsda sud tizimi qanday ishlaydi?
JAVOB: Berilgan hujjatlarda bu haqda ma'lumot topilmadi.
MANBA: ['PF-140.txt']
--------------

## Bosqich 7 — o'z hujjatlaringiz bilan ishlash (qisqa yo'riqnoma)

Demonstratsiya hujjatini (PF-140) o'z hujjatlaringizga almashtirish uchun:

1. **Hujjatlaringizni tayyorlang.** Fayl `.txt` yoki `.md` formatida bo'lishi kerak. Agar hujjatingiz Word yoki PDF formatida bo'lsa, avval uni matn (`.txt`) formatiga saqlang (Word'da: **Fayl → Saqlash → Plain Text**).
2. **Yuklang.** Colab'ning chap tomonidagi 📁 (Files) panelida `documents` papkasini oching va fayl(lar)ingizni shu yerga sudrab tashlang. Xohlasangiz, PF-140.txt'ni o'chiring yoki qoldiring — bir nechta hujjat bir vaqtda ishlatilishi mumkin.
3. **Qayta ishga tushiring.** Notebookni **yuqoridan boshlab** (Bosqich 0'dan) ketma-ket, birma-bir ishga tushiring — *Runtime → Run all* ham ishlaydi. Bu muhim: Bosqich 2 bazani tozalab, hujjatlaringiz asosida qayta quradi.
4. **Savol bering.** Bosqich 5'dagi (yoki yangi katakdagi) `answer_question("...")` funksiyasiga o'z savolingizni yozing.

**Parametrlarni sozlash kerak bo'lsa** (Bosqich 0'dagi `CHUNK_SIZE`, `CHUNK_OVERLAP`, `TOP_K`): agar tizim ma'lum javobga "topilmadi" desa, avval `TOP_K`ni oshirib ko'ring (masalan, 8), yordam bermasa — `CHUNK_SIZE`ni kichraytiring (masalan, 200). Har o'zgarishdan keyin **Bosqich 2'dan** qayta ishga tushirish shart.

---
## Muallif va litsenziya

© 2026, Bunyod Panjiyev. Barcha huquqlar himoyalangan.

Ushbu notebook BePro Academy'ning "Sun'iy intellekt va mashinali o'qitish asoslari" kursi doirasida,
ta'lim maqsadida yaratilgan (RAG-EDU-1 texnik topshirig'i asosida).

Namunaviy hujjat sifatida ishlatilgan PF-140 (2025-yil 21-avgustdagi Prezident Farmoni)
ochiq davlat manbasidan olingan.

In [18]:


!pip install -q gradio

import gradio as gr

def ask(question):
    answer, sources = answer_question(question, verbose=False)
    sources_text = ", ".join(sources) if sources else "—"
    return f"{answer}\n\n📄 Manba(lar): {sources_text}"

demo = gr.Interface(
    fn=ask,
    inputs=gr.Textbox(
        label="Savolingizni yozing",
        placeholder="Masalan: Iqtisodiy sudlar tuzilmasi qanday optimallashtiriladi?",
        lines=2,
    ),
    outputs=gr.Textbox(label="Javob"),
    title="Yuridik Hujjatlar bo'yicha AI Yordamchi",
    description=(
        "Bu tizim yuklangan yuridik hujjatlar (masalan, PF-140 farmoni) asosida "
        "savollaringizga javob beradi. Savolingizni yozing va pastdagi 'Submit' "
        "tugmasini bosing."
    ),
    examples=[
        "Iqtisodiy sudlar tuzilmasi qanday optimallashtiriladi?",
        "Toshkent shahrida qachon 'Raqamli sud' zallari tashkil etiladi?",
        "Sud majlisi bayonnomalari qanday tayyorlanadi?",
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a8041408a8cd7679e2.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
